## Import

In [7]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [21]:
from sindex.metrics.citations import merge_citations_dicts, merge_citations_from_files, merge_citations_from_files_fast
import os

## Citations

### Merge citations from different sources

#### DataCite (DOIs)

In [12]:
mdc_citations = r"I:\pipeline-data\citations\mdc\mdc_citations.ndjson"
oa_citations = r"I:\pipeline-data\citations\openalex\oa_citations.ndjson"
dc_citations = ""
citation_files = [mdc_citations, oa_citations, dc_citations]
output_file_doi = r"I:\pipeline-data\citations\doi_citations.ndjson"

In [13]:
merge_citations_from_files_fast(citation_files, output_file_doi)

Starting merge of 2 valid files...
Finished processing 4,220,316 records. Unique: 4,141,376
Writing to I:\pipeline-data\citations\doi_citations.ndjson...
Done!
Done!


#### EMDB

In [17]:
mdc_citations = r"I:\pipeline-data\citations\mdc\mdc_citations_emdb.ndjson"
citation_files = [mdc_citations]
output_file_emdb = r"I:\pipeline-data\citations\emdb_citations.ndjson"

In [18]:
merge_citations_from_files_fast(citation_files, output_file_emdb)

Starting merge of 1 valid files...
Finished processing 15,134 records. Unique: 15,134
Writing to I:\pipeline-data\citations\emdb_citations.ndjson...
Done!


### Combine

In [19]:
doi = r"I:\pipeline-data\citations\doi_citations.ndjson"
emdb = r"I:\pipeline-data\citations\emdb_citations.ndjson"
file_list = [doi, emdb]
output_path = r"I:\pipeline-data\citations\citations.ndjson"

In [22]:
line_count = 0

with open(output_path, 'w', encoding='utf-8') as outfile:
    for file_path in file_list:
        if os.path.exists(file_path):
            print(f"Processing: {os.path.basename(file_path)}...")
            with open(file_path, 'r', encoding='utf-8') as infile:
                for line in infile:
                    # Strip extra whitespace/newlines to ensure one object per line
                    clean_line = line.strip()
                    if clean_line:
                        outfile.write(clean_line + '\n')
                        line_count += 1
        else:
            print(f"Warning: File not found: {file_path}")

print(f"Done! Combined {len(file_list)} files. Total rows: {line_count:,}")

Processing: doi_citations.ndjson...
Processing: emdb_citations.ndjson...
Done! Combined 2 files. Total rows: 4,156,510


## Mentions

In [23]:
# Mock mentionds file
input_file =  r"I:\pipeline-data\citations\citations.ndjson"
output_file =  r"I:\pipeline-data\mentions\mentions.ndjson"

with open(input_file, 'r') as infile, open(output_file, 'w') as outfile:
    for line in infile:
        if not line.strip(): continue  # Skip empty lines
        data = json.loads(line)
        
        # Mapping logic: Rename only if key exists
        mapping = {
            'citation_link': 'mention_link',
            'citation_date': 'mention_date',
            'citation_weight': 'mention_weight'
        }
        
        for old_key, new_key in mapping.items():
            if old_key in data:
                data[new_key] = data.pop(old_key)
        
        outfile.write(json.dumps(data) + '\n')

In [ ]:
# Create consolidated dataset

## Normalization factors